# target encoding 深掘り

ターゲットエンコーディングを CV 内で正しく適用し、平滑化でリークと過学習を抑える方法を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/advanced/01-target-encoding`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Kaggle Notebook 上では既存の /kaggle/input/titanic をそのまま使用し、
# Google Colab やローカル環境では互換スキーマの Titanic サンプルデータ（891行 / 418行）を自動配置します。
# ※ Kaggle 本番データで提出する場合は kaggle CLI (!kaggle competitions download -c titanic) をご利用ください。
import os
import numpy as np
import pandas as pd

def _ensure_titanic_data():
    target_dir = "/kaggle/input/titanic"
    if os.path.exists(os.path.join(target_dir, "train.csv")):
        return

    try:
        os.makedirs(target_dir, exist_ok=True)
        data_dir = target_dir
    except OSError:
        data_dir = os.path.abspath("./.kaggle_input/titanic")
        os.makedirs(data_dir, exist_ok=True)
        _orig_read_csv = pd.read_csv

        def _compat_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/"):
                rel = filepath_or_buffer[len("/kaggle/input/"):]
                local_path = os.path.abspath(os.path.join("./.kaggle_input", rel))
                if os.path.exists(local_path):
                    filepath_or_buffer = local_path
            return _orig_read_csv(filepath_or_buffer, *args, **kwargs)

        pd.read_csv = _compat_read_csv

    rng = np.random.default_rng(42)
    surnames = ["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen", "Moran", "McCarthy", "Palsson", "Johnson", "Nasser"]
    given_m = ["Owen", "William", "James", "Henry", "Leonard", "Timothy", "charles", "George"]
    given_f = ["Florence", "Laina", "Lily", "elisabeth", "Adele", " Marguerite", "Anna", "Mary"]

    def _make_split(n, start_id, with_target):
        pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
        sex = rng.choice(["male", "female"], size=n, p=[0.65, 0.35])
        age = np.clip(rng.normal(29.7, 14.5, size=n), 0.42, 80.0).round(1)
        sibsp = rng.choice([0, 1, 2, 3, 4], size=n, p=[0.68, 0.23, 0.05, 0.02, 0.02])
        parch = rng.choice([0, 1, 2, 3], size=n, p=[0.76, 0.13, 0.09, 0.02])
        base_fare = np.where(pclass == 1, 60.0, np.where(pclass == 2, 20.0, 11.0))
        fare = (base_fare * rng.lognormal(0.0, 0.55, size=n)).round(2)
        embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.19, 0.09]).astype(object)

        names = []
        cabins = []
        tickets = []
        for i in range(n):
            sn = surnames[i % len(surnames)]
            if sex[i] == "male":
                title = "Master" if age[i] < 13 else "Mr"
                gn = given_m[i % len(given_m)].strip().capitalize()
            else:
                title = "Miss" if age[i] < 24 else "Mrs"
                gn = given_f[i % len(given_f)].strip().capitalize()
            names.append(f"{sn}, {title}. {gn}")
            cabins.append(f"{'ABC'[pclass[i] - 1]}{10 + (i % 90)}" if rng.random() > 0.77 else np.nan)
            tickets.append(f"PC {17500 + (i % 350)}" if pclass[i] == 1 else str(230000 + i))

        age_obj = age.astype(float)
        age_obj[rng.random(n) < 0.20] = np.nan
        embarked[rng.random(n) < 0.005] = np.nan
        if not with_target:
            fare[0] = np.nan

        data = {
            "PassengerId": np.arange(start_id, start_id + n),
            "Pclass": pclass,
            "Name": names,
            "Sex": sex,
            "Age": age_obj,
            "SibSp": sibsp,
            "Parch": parch,
            "Ticket": tickets,
            "Fare": fare,
            "Cabin": pd.Series(cabins, dtype=object),
            "Embarked": pd.Series(embarked, dtype=object),
        }
        if with_target:
            family_size = sibsp + parch + 1
            logit = (
                -1.65
                + np.where(sex == "female", 2.45, 0.0)
                + np.where(pclass == 1, 1.35, np.where(pclass == 2, 0.35, -0.55))
                + np.where(np.nan_to_num(age_obj, nan=29.0) < 12, 1.2, 0.0)
                + np.where((family_size >= 2) & (family_size <= 4), 0.75, -0.35)
            )
            prob = 1.0 / (1.0 + np.exp(-logit))
            survived = (rng.random(n) < prob).astype(int)
            ordered = {"PassengerId": data["PassengerId"], "Survived": survived}
            for k, v in data.items():
                if k != "PassengerId":
                    ordered[k] = v
            return pd.DataFrame(ordered)
        return pd.DataFrame(data)

    _make_split(891, 1, True).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    _make_split(418, 892, False).to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"Titanic dataset ready at: {data_dir}")

_ensure_titanic_data()

In [ ]:
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

## このトラックで学ぶこと

このトラック（L8）では、上位入賞・グランドマスターを目指すための**上級テクニック**と、
**運用・チーム戦略・称号獲得**の実戦知見を体系化します。L7 までの土台の上に、勝負を分ける一手を積みます。

## target encoding の威力と危うさ

ターゲットエンコーディングは、カテゴリ変数を**目的変数の統計量**（多くは平均）で数値化する強力な手法です。
高カーディナリティでも効きますが、**目的変数を使う**ため最もリークしやすいのが弱点です。

## fold 内（out-of-fold）で適用する

リークを防ぐ原則は単純です。各サンプルのエンコード値は、**そのサンプルを含まない**データから計算します。

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

def target_encode_oof(train, test, col, target, n_splits=5, smoothing=10):
    oof = np.zeros(len(train))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr, va in skf.split(train, train[target]):
        prior_tr = train.iloc[tr][target].mean()
        agg = train.iloc[tr].groupby(col)[target].agg(["mean", "count"])
        # 平滑化: 件数が少ない水準は fold 内の事前平均(prior_tr)へ寄せる
        smooth = (agg["mean"] * agg["count"] + prior_tr * smoothing) / (agg["count"] + smoothing)
        oof[va] = train.iloc[va][col].map(smooth).fillna(prior_tr).values
    # test は train 全体で計算した平滑化値を使う
    prior = train[target].mean()
    agg = train.groupby(col)[target].agg(["mean", "count"])
    smooth = (agg["mean"] * agg["count"] + prior * smoothing) / (agg["count"] + smoothing)
    test_enc = test[col].map(smooth).fillna(prior).values
    return oof, test_enc

In [ ]:
# Titanic の Embarked 列に OOF Target Encoding を適用して結果を確認します
oof_te, test_te = target_encode_oof(train, test, col="Embarked", target="Survived")
print("OOF encoded Embarked (first 5):", oof_te[:5].round(4))
print("Test encoded Embarked (first 5):", test_te[:5].round(4))

## 平滑化（smoothing）の意味

出現回数の少ない水準は、その平均が偏りやすく過学習の原因になります。
**件数に応じて全体平均へ寄せる**（平滑化）ことで、安定したエンコードになります。

> **Warning:** 全データでターゲット平均を計算して付与すると、検証側の答えが漏れるデータリークになり、 CV だけ高くて本番で崩れます。必ず OOF で作り、交差検証で効果を確認します。

## 次のステップ

次は、train/test の分布差を検出する adversarial validation を学びます。